In [0]:
from pathlib import Path
from pyspark.sql.functions import current_timestamp, lit

CATALOGO = "will"
ESQUEMA = "i30"

pasta_dados = Path(
    "/Workspace/Users/willdeglan@gmail.com/Projetos/i30/datasource"
)

arquivos = {
    "viagens": "registro_viagens.csv",
    "caronas": "registro_caronas.csv",
    "abastecimentos": "registro_abastecimento.csv",
}

tabelas_bronze = {
    "viagens": "bronze_registro_viagens",
    "caronas": "bronze_registro_caronas",
    "abastecimentos": "bronze_registro_abastecimento",
}

# O catálogo "will" precisa existir e você precisa ter permissão para usá-lo.
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOGO}.{ESQUEMA}")

dados = {}

for nome, arquivo in arquivos.items():
    caminho = pasta_dados / arquivo

    if not caminho.is_file():
        raise FileNotFoundError(f"CSV não encontrado: {caminho}")

    dados[nome] = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "false")
        .option("encoding", "UTF-8")
        .csv(f"file:{caminho}")
    )

for nome, tabela in tabelas_bronze.items():
    df_bronze = (
        dados[nome]
        .withColumn("_ingested_at", current_timestamp())
        .withColumn("_source_file", lit(arquivos[nome]))
    )

    (
        df_bronze.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{CATALOGO}.{ESQUEMA}.{tabela}")
    )

    print(f"Tabela gravada: {CATALOGO}.{ESQUEMA}.{tabela}")

In [0]:
def sql_texto(texto):
    """Escapa texto para uso como literal SQL."""
    return "'" + texto.replace("'", "''") + "'"


comentarios = {
    "bronze_registro_viagens": {
        "tabela": "Camada Bronze com os registros brutos de viagens do veículo, lidos do CSV do GitHub.",
        "colunas": {
            "DATA": "Data do registro da viagem.",
            "INICIAL": "Quilometragem do odômetro no início do percurso, em km.",
            "FINAL": "Quilometragem do odômetro no final do percurso, em km.",
            "PAINEL_L_100KM": "Consumo médio indicado pelo painel do veículo, em litros por 100 km.",
            "KM_H": "Velocidade média registrada para o percurso, em km/h.",
            "_ingested_at": "Data e hora em que o registro foi carregado na camada Bronze.",
            "_source_file": "Nome do arquivo CSV de origem do registro.",
        },
    },
    "bronze_registro_caronas": {
        "tabela": "Camada Bronze com os registros brutos de passageiros e ganhos com caronas, lidos do CSV do GitHub.",
        "colunas": {
            "DATA": "Data do registro das caronas.",
            "MANHA_QTD": "Quantidade de passageiros transportados no período da manhã.",
            "MANHA_RS": "Valor recebido com caronas no período da manhã, em reais.",
            "NOITE_QTD": "Quantidade de passageiros transportados no período da noite.",
            "NOITE_RS": "Valor recebido com caronas no período da noite, em reais.",
            "_ingested_at": "Data e hora em que o registro foi carregado na camada Bronze.",
            "_source_file": "Nome do arquivo CSV de origem do registro.",
        },
    },
    "bronze_registro_abastecimento": {
        "tabela": "Camada Bronze com os registros brutos de abastecimentos do veículo, lidos do CSV do GitHub.",
        "colunas": {
            "DATA": "Data do abastecimento.",
            "R$_LITRO": "Preço cobrado por litro de combustível, em reais por litro.",
            "R$_TOTAL": "Valor total pago no abastecimento, em reais.",
            "LITROS": "Quantidade de combustível abastecida, em litros.",
            "_ingested_at": "Data e hora em que o registro foi carregado na camada Bronze.",
            "_source_file": "Nome do arquivo CSV de origem do registro.",
        },
    },
}

for tabela, definicao in comentarios.items():
    tabela_completa = f"`{CATALOGO}`.`{ESQUEMA}`.`{tabela}`"

    spark.sql(
        f"COMMENT ON TABLE {tabela_completa} "
        f"IS {sql_texto(definicao['tabela'])}"
    )

    for coluna, comentario in definicao["colunas"].items():
        coluna_sql = f"`{coluna.replace('`', '``')}`"

        spark.sql(
            f"COMMENT ON COLUMN {tabela_completa}.{coluna_sql} "
            f"IS {sql_texto(comentario)}"
        )

    print(f"Descrição e comentários aplicados: {CATALOGO}.{ESQUEMA}.{tabela}")

In [0]:
for tabela in tabelas_bronze.values():
    nome_completo = f"{CATALOGO}.{ESQUEMA}.{tabela}"
    tabela_sql = f"`{CATALOGO}`.`{ESQUEMA}`.`{tabela}`"

    print(f"\n--- Dados: {nome_completo} ---")
    display(spark.table(nome_completo))

    print(f"\n--- Descrição e comentários: {nome_completo} ---")
    display(spark.sql(f"DESCRIBE TABLE EXTENDED {tabela_sql}"))